In [12]:
!pip install spacy
!python -m spacy download en_core_web_sm
from tokenizer import TinyStoriesTokenizer
from tqdm import tqdm
import numpy as np
import json
import spacy
import torch
import random


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 123.0 MB/s  0:00:00

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [4]:
from torch.utils.data import Dataset, DataLoader
from tokenizer import TinyStoriesTokenizer

class Config :
    vocab_size: int = 5000  # This number should agree with the tokenizer
    number_of_transformer_blocks: int = 4
    number_of_attention_heads: int = 4
    vector_dim: int = 64
    block_size: int = 512
    dropout_prob: float = 0.1
    batch_size: int = 8
    learning_rate: float = 0.0005
    weight_decay: float = 0.000001
    no_of_epochs: int = 1


class TinyStoriesDataset(Dataset):
    def __init__(self, data_file, block_size):
        """
        data_file: path to the .bin file (uint16 array of token IDs)
        block_size: the context window (e.g., 256 or 512 tokens)
        """

        # Memory-map the data file (RAM usage stays near zero!)
        self.data = np.memmap(data_file, dtype=np.uint16, mode='r')
        self.block_size = block_size

    def __len__(self):
        # We subtract block_size to ensure we don't go out of bounds
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # Pull a chunk of length block_size + 1 (data and target)
        chunk = self.data[idx : idx + self.block_size + 1]

        # Convert to torch tensors
        x = torch.from_numpy(chunk[:-1].astype(np.int64)) # Input
        y = torch.from_numpy(chunk[1:].astype(np.int64))  # Target (shifted by 1)

        return x, y 


In [10]:
def get_POS_tags(text):
    doc = nlp(text)
    tags_list = []
    
    for token in doc:
        tags_list.append({
            "word": token.text,
            "pos": token.pos_
        })
    return tags_list # [{"word": "The", "pos": "DET"},...

def create_subset_POS(training_dataset, tokenizer, n_subset = 5_000):
    """
    Saves original text and Spacy POS, doesn't matter how the
    bpe tonkenizer splits words
    """
    subset = []
    nlp = spacy.load("en_core_web_sm")
    
    
    random.seed(42)
    sample_idx = random.sample(range(len(training_dataset)), n_subset)
    
    print(f"Processing {n_subset} stories ...")
    for idx in tqdm(sample_idx, desc="Processing stories"):
        x_sample, _ = training_dataset[idx]
        tokens_x = x_sample.tolist() 
        full_text = "".join(tokenizer.vocab[i] for i in tokens_x)

        dicc_pos = get_POS_tags(full_text)
        subset.append({
            "story_idx": idx,
            "text": full_text,
            "annotations": dicc_pos})
    
    #print('subset', subset[0])
    return subset


In [13]:
training_dataset = TinyStoriesDataset('/datasets/dd2417/train.bin', Config.block_size)
tokenizer = TinyStoriesTokenizer.load('/datasets/dd2417/tokenizer.json')
#subset = create_subset_POS(training_dataset, tokenizer, n_subset = 5_000)
 
#with open('subset_pos.json', 'w') as f:
 #   json.dump(subset, f)


In [6]:
with open('subset_pos.json', 'r') as f:
    subset = json.load(f)

In [7]:
##### ALIGN POS WITH BPE ########

aligned_data = []
for item in subset:
    _, transformer_token_ids = tokenizer.tokenize(item['text'])
    annotations = item['annotations']
    
    aligned_tags = ["IGNORE"] * len(transformer_token_ids)
    current_token = 0
    
    for dicc in annotations:
        word = dicc['word']
        pos_tag = dicc['pos']

        # We dont care about spaces in SpaCy. BPE already included them in the words ' funny'
        # tokenize current word into subtokens
        if pos_tag == "SPACE" or word.strip() == "":
            continue

        subtokens, subtokens_idx = tokenizer.tokenize(word.strip()) #erase initial space ' funny'
        num_subtokens = len(subtokens_idx)
        # print('subtokens-', dicc, subtokens)
        # Last part - assign tag to last subtoken
        # Assign Spacy only to that last subtoken

        if num_subtokens > 0:
            last_idx = current_token + num_subtokens - 1
            if last_idx < len(aligned_tags):
                aligned_tags[last_idx] = pos_tag
            current_token += num_subtokens
        #print('aligned_tags:', aligned_tags[0:current_token])
    

    aligned_data.append({
        "input_ids": transformer_token_ids, # numbers representing tokens in BPE [1089, 1045, 1365...]
        "pos_tags": aligned_tags  # POS name aligned token by token bpe: ["IGNORE", "NOUN", "VERB"---]
    }) 

    """
    [{'input_ids': [1089, 1045, 1365], 'pos_tags': ['IGNORE', 'ADJ', ...]}


    """

In [8]:
with open("aligned_data.json", "w", encoding="utf-8") as f:
    json.dump(aligned_data, f)


In [14]:
with open('aligned_data.json', 'r') as f:
    aligned_data = json.load(f)

In [16]:
"""
Dictionary relationship between POS tags and idx.
"""

unique_tags = set()
for item in aligned_data:
    for tag in item['pos_tags']:
        if tag != "IGNORE":
            unique_tags.add(tag)

# Map Tag -> ID
# order so its always the same
sorted_tags = sorted(list(unique_tags))
pos_to_id = {tag: i for i, tag in enumerate(sorted_tags)}
id_to_pos =  {v: k for k, v in pos_to_id.items()}

# Save mapping
with open("aligned_data_metadata.json", "w") as f:
    json.dump({
        "pos_to_id": pos_to_id,
        "id_to_pos": id_to_pos,
        "ignore_index": -100 #for pytorch ignore
              }, f, indent=4)

print(f"Dicc with {len(pos_to_id)} categories")
print(pos_to_id)

Dicc with 17 categories
{'ADJ': 0, 'ADP': 1, 'ADV': 2, 'AUX': 3, 'CCONJ': 4, 'DET': 5, 'INTJ': 6, 'NOUN': 7, 'NUM': 8, 'PART': 9, 'PRON': 10, 'PROPN': 11, 'PUNCT': 12, 'SCONJ': 13, 'SYM': 14, 'VERB': 15, 'X': 16}


In [ ]:
## Save pos tags as numbers to be used before training 
# and use category diccionary to go back to word 


with open("aligned_data_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)
    pos_to_id = metadata["pos_to_id"]
    ignore_index = metadata["ignore_index"]


with open("aligned_data.json", "r", encoding="utf-8") as f:
    raw_aligned_data = json.load(f)

dataset_final = []

for item in raw_aligned_data:
    numeric_tags = [
        pos_to_id[tag] if tag != "IGNORE" else ignore_index 
        for tag in item['pos_tags']
    ]
    
    dataset_final.append({
        "input_ids": item['input_ids'],
        "label": numeric_tags })